# Ideal Knowledge Graph Extraction Trainer

This notebook trains a **Unified Neural Backbone** for high-throughput Entity and Relationship extraction using a Bi-Encoder architecture.

### Key Features:
1. **Backbone**: Jina-v3 (8k context, task-specific adapters).
2. **NER Head**: Bi-Encoder mapping text spans to Wikidata label embeddings.
3. **RE Head**: GLiDRE-style head with **Localized Context Pooling** to connect distant entities.
4. **Training**: Alternating task batches with **InfoNCE Contrastive Loss**.

In [ ]:
!pip install -q torch transformers datasets accelerate sentencepiece

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, AdamW
from tqdm.auto import tqdm
import numpy as np

## 1. The Unified Multi-Head Model
This model uses a shared Jina-v3 backbone with specialized projection heads.

In [ ]:
class UnifiedKGModel(nn.Module):
    def __init__(self, model_name='jinaai/jina-embeddings-v3', label_dim=768):
        super().__init__()
        # Load Jina v3 (Requires trust_remote_code=True)
        self.backbone = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        self.hidden_size = self.backbone.config.hidden_size

        # NER Head: Projects token to label space
        self.ner_projection = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.GELU(),
            nn.Linear(self.hidden_size, label_dim)
        )

        # RE Head: Projects (Subj + Obj + Context) to relation space
        self.re_projection = nn.Sequential(
            nn.Linear(self.hidden_size * 3, self.hidden_size),
            nn.GELU(),
            nn.Linear(self.hidden_size, label_dim)
        )

        self.label_dim = label_dim
        self.temperature = nn.Parameter(torch.ones([]) * 0.07)


    def forward_backbone(self, input_ids, attention_mask):
        # Jina v3 returns (last_hidden_state, attentions, etc.)
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, output_attentions=True)
        return outputs.last_hidden_state, outputs.attentions[-1] # Use last layer attention


    def get_ner_score(self, token_embeddings, span_idx):
        # Extract specific span token and project
        span_vec = token_embeddings[torch.arange(token_embeddings.size(0)), span_idx]
        return F.normalize(self.ner_projection(span_vec), p=2, dim=-1)


    def get_re_score(self, token_embeddings, attentions, subj_idx, obj_idx):
        batch_size = token_embeddings.size(0)
        b_idx = torch.arange(batch_size)

        # 1. Entity Vectors
        s_vec = token_embeddings[b_idx, subj_idx]
        o_vec = token_embeddings[b_idx, obj_idx]

        # 2. Localized Context Pooling (ATLOP)
        # Calculate shared attention: (batch, num_heads, seq_len)
        # attn: (batch, num_heads, seq, seq)
        attn_s = attentions[b_idx, :, subj_idx, :]
        attn_o = attentions[b_idx, :, obj_idx, :]

        # Shared focus across all tokens
        shared_attn = (attn_s * attn_o).sum(dim=1) # Sum over heads
        ctx_weights = F.softmax(shared_attn, dim=-1)

        # Context = Weighted sum of all hidden states
        ctx_vec = torch.bmm(ctx_weights.unsqueeze(1), token_embeddings).squeeze(1)

        # 3. Combine and Project
        combined = torch.cat([s_vec, o_vec, ctx_vec], dim=-1)
        return F.normalize(self.re_projection(combined), p=2, dim=-1)

## 2. Dataset Implementation (Span Alignment)
This handles the character-to-token mapping for Pile-NER and Re-DocRED.

In [ ]:
class UnifiedKGDataset(Dataset):
    def __init__(self, data, tokenizer, label_db, task='NER'):
        self.data = data
        self.tokenizer = tokenizer
        self.label_db = label_db
        self.task = task

    def __len__(self):
        return len(self.data)

    def find_token_idx(self, char_idx, offsets):
        for i, (start, end) in enumerate(offsets):
            if start <= char_idx < end:
                return i
        return 0

    def __getitem__(self, idx):
        item = self.data[idx]
        encoding = self.tokenizer(item['text'], truncation=True, max_length=512, return_offsets_mapping=True, return_tensors='pt')

        res = {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label_vec': self.label_db.get(item['label'], torch.zeros(768))
        }

        offsets = encoding['offset_mapping'].squeeze(0)
        if self.task == 'NER':
            res['span_idx'] = self.find_token_idx(item['start'], offsets)
        else:
            res['subj_idx'] = self.find_token_idx(item['subj_start'], offsets)
            res['obj_idx'] = self.find_token_idx(item['obj_start'], offsets)

        return res


def collate_fn(batch):
    ids = torch.nn.utils.rnn.pad_sequence([x['input_ids'] for x in batch], batch_first=True, padding_value=0)
    mask = torch.nn.utils.rnn.pad_sequence([x['attention_mask'] for x in batch], batch_first=True, padding_value=0)
    labels = torch.stack([x['label_vec'] for x in batch])

    out = {'input_ids': ids, 'attention_mask': mask, 'labels': labels}
    if 'span_idx' in batch[0]:
        out['span_idx'] = torch.tensor([x['span_idx'] for x in batch])
    else:
        out['subj_idx'] = torch.tensor([x['subj_idx'] for x in batch])
        out['obj_idx'] = torch.tensor([x['obj_idx'] for x in batch])
        return out

## 3. Training Logic (Contrastive Loss)
Teaching the model to match text to Wikidata definitions.

In [ ]:
def info_nce_loss(predictions, targets, temp=0.07):
    # Batch Matrix Multiplication
    logits = torch.matmul(predictions, targets.T) / temp
    labels = torch.arange(predictions.size(0)).to(predictions.device)
    return F.cross_entropy(logits, labels)

def train_loop(model, ner_loader, re_loader, epochs=3, device='cuda'):
    model.to(device)
    # Freeze backbone for initial head training
    for param in model.backbone.parameters(): param.requires_grad = False

    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        # Alternate Tasks
        pbar = tqdm(zip(ner_loader, re_loader), total=min(len(ner_loader), len(re_loader)))
        for ner_batch, re_batch in pbar:
            optimizer.zero_grad()

            # NER Step
            states, attns = model.forward_backbone(ner_batch['input_ids'].to(device), ner_batch['attention_mask'].to(device))
            ner_preds = model.get_ner_score(states, ner_batch['span_idx'].to(device))
            ner_loss = info_nce_loss(ner_preds, ner_batch['labels'].to(device))

            # RE Step
            states, attns = model.forward_backbone(re_batch['input_ids'].to(device), re_batch['attention_mask'].to(device))
            re_preds = model.get_re_score(states, attns, re_batch['subj_idx'].to(device), re_batch['obj_idx'].to(device))
            re_loss = info_nce_loss(re_preds, re_batch['labels'].to(device))

            loss = ner_loss + re_loss
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            pbar.set_description(f\"Epoch {epoch} | Loss: {total_loss/(pbar.n+1):.4f}\")